# Task 2：单只银行定投回测

对应 [2-单只银行定投回测.md](./2-单只银行定投回测.md)。

以工商银行 601398 为例，执行 10Y/5Y/3Y 月度定投回测，输出交易流水、9 组指标和 5 张图。

## 1. 环境与参数

回测口径：每月 5,000 元，1 日买入，100 股整数倍，分红再投资，不计手续费和分红税。

In [ ]:
from __future__ import annotations

import importlib
import math
import os
import re
import socket
import sys
import time
from contextlib import contextmanager
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Iterable
from unittest.mock import patch

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


WORKING_DIR = Path.cwd().resolve()
LABS_DIR = None
for _candidate in [WORKING_DIR, *WORKING_DIR.parents]:
    if _candidate.name == "labs" and (_candidate / "pyproject.toml").is_file():
        LABS_DIR = _candidate
        break
    if (_candidate / "labs" / "pyproject.toml").is_file():
        LABS_DIR = _candidate / "labs"
        break

LAB_DIR = LABS_DIR / "01_银行股定投回测"


plt.rcParams["font.sans-serif"] = [
    "Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"
]
plt.rcParams["axes.unicode_minus"] = False


SYMBOL = "601398"
NAME = "工商银行"
AS_OF_DATE = "2026-07-31"
HORIZONS = (10, 5, 3)
MONTHLY_AMOUNT = 5000
BUY_DAY = 1
LOT_SIZE = 100
DIVIDEND_REINVEST = True

DATA_DIR = LAB_DIR / "data" / "lab2"
PRICE_DIR = DATA_DIR / "prices"
DIVIDEND_DIR = DATA_DIR / "dividends"
CHART_DIR = LAB_DIR / "data" / "charts" / "task2_single"
RESULT_DIR = LAB_DIR / "data" / "results"
TRANSACTION_DIR = RESULT_DIR / "transactions"
for path in (CHART_DIR, RESULT_DIR, TRANSACTION_DIR):
    path.mkdir(parents=True, exist_ok=True)

print(f"标的: {SYMBOL} {NAME}")
print(f"回测截止日: {AS_OF_DATE}")
print(f"周期: {HORIZONS}")
print(f"月度投入: {MONTHLY_AMOUNT} 元")

## 2. 函数定义

以下函数从原 `bank_core.py` 内联到本 Notebook，实现回测核心逻辑。

### 回测计算函数

本组函数实现月度定投回测的核心逻辑：构建标的总收益净值、计算 XIRR、执行按月买入和分红再投资。

- `normalize_symbol`：从任意输入提取 6 位证券代码，前补零。
- `build_total_return_history`：用不复权收盘价 + 每股现金分红构建标的总收益净值（用于回撤计算）。
- `xirr`：计算 XIRR（内部收益率），牛顿迭代法，容忍不规则现金流日期。
- `contribution_dates`：计算定投买入日（每月 1 日，非交易日顺延至下一个交易日）。
- `_shares_on_or_before`：查找登记日对应的持股快照（用于分红再投资时的持股数）。
- `_max_loss_duration_days`：账户资产低于累计投入的最长连续天数。
- `_strategy_max_drawdown`：现金流调整后策略净值（含分红再投资）的最大回撤。
- `BacktestOutput`：数据类，封装回测结果（summary/transactions/account_history/total_return_history）。
- `simulate_bank_dca`：单只银行月度定投回测主函数：按月买入、分红再投资、计算 9 组指标。

In [ ]:
def normalize_symbol(value: Any) -> str:
    digits = "".join(c for c in str(value) if c.isdigit())
    return digits[-6:].zfill(6)


def build_total_return_history(
    prices: pd.DataFrame,
    dividends: pd.DataFrame,
) -> pd.DataFrame:
    """用不复权收盘价和每股现金分红构建标的总收益净值。"""
    frame = prices[["date", "close"]].copy()
    frame["date"] = pd.to_datetime(frame["date"])
    frame = frame.dropna().drop_duplicates("date", keep="last").sort_values("date")
    events = dividends[["ex_date", "cash_dividend_per_share"]].copy()
    events["ex_date"] = pd.to_datetime(events["ex_date"])
    events = events.groupby("ex_date", as_index=False)["cash_dividend_per_share"].sum()
    frame = frame.merge(events, left_on="date", right_on="ex_date", how="left")
    frame["cash_dividend_per_share"] = frame["cash_dividend_per_share"].fillna(0.0)
    frame["daily_total_return"] = (
        (frame["close"] + frame["cash_dividend_per_share"])
        / frame["close"].shift(1)
        - 1.0
    )
    frame.loc[frame.index[0], "daily_total_return"] = 0.0
    frame["total_return_nav"] = (1.0 + frame["daily_total_return"]).cumprod()
    frame["drawdown"] = (
        frame["total_return_nav"] / frame["total_return_nav"].cummax() - 1.0
    )
    return frame[
        [
            "date", "close", "cash_dividend_per_share", "daily_total_return",
            "total_return_nav", "drawdown",
        ]
    ]


def xirr(cashflows: Iterable[float], dates: Iterable[Any]) -> float:
    values = np.asarray(list(cashflows), dtype=float)
    timestamps = [pd.Timestamp(date) for date in dates]
    if len(values) != len(timestamps) or len(values) < 2:
        return np.nan
    if np.all(values >= 0) or np.all(values <= 0):
        return np.nan
    years = np.array(
        [(date - timestamps[0]).total_seconds() / (365.25 * 86400)
         for date in timestamps],
        dtype=float,
    )

    def npv(rate: float) -> float:
        return float(np.sum(values / np.power(1.0 + rate, years)))

    lower = -0.9999
    upper = 1.0
    lower_value = npv(lower)
    upper_value = npv(upper)
    while lower_value * upper_value > 0 and upper < 1_000_000:
        upper = upper * 2.0 + 1.0
        upper_value = npv(upper)
    if lower_value * upper_value > 0:
        return np.nan
    for _ in range(250):
        midpoint = (lower + upper) / 2.0
        midpoint_value = npv(midpoint)
        if abs(midpoint_value) < 1e-8:
            return float(midpoint)
        if lower_value * midpoint_value <= 0:
            upper = midpoint
        else:
            lower = midpoint
            lower_value = midpoint_value
    return float((lower + upper) / 2.0)


def contribution_dates(
    trading_dates: Iterable[Any],
    *,
    start_date: Any,
    end_date: Any,
    buy_day: int = 1,
) -> list[pd.Timestamp]:
    dates = pd.DatetimeIndex(pd.to_datetime(list(trading_dates))).sort_values().unique()
    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)
    dates = dates[(dates >= start) & (dates <= end)]
    if len(dates) == 0:
        return []

    selected: list[pd.Timestamp] = [pd.Timestamp(dates[0])]
    first_period = pd.Timestamp(dates[0]).to_period("M")
    last_period = pd.Timestamp(dates[-1]).to_period("M")
    for period in pd.period_range(first_period + 1, last_period, freq="M"):
        target = period.start_time + pd.Timedelta(days=buy_day - 1)
        candidates = dates[
            (dates.to_period("M") == period) & (dates >= target)
        ]
        if len(candidates):
            selected.append(pd.Timestamp(candidates[0]))
    return selected


def _shares_on_or_before(
    snapshots: dict[pd.Timestamp, int],
    date: pd.Timestamp,
) -> int:
    eligible = [key for key in snapshots if key <= date]
    return snapshots[max(eligible)] if eligible else 0


def _max_loss_duration_days(account_history: pd.DataFrame) -> int:
    """账户资产低于累计投入的最长连续天数。"""
    if account_history.empty:
        return 0
    asset = account_history["account_asset"].astype(float).reset_index(drop=True)
    contribution = account_history["cumulative_contribution"].astype(float).reset_index(drop=True)
    underwater = (asset < contribution).fillna(False)
    if not underwater.any():
        return 0
    max_run = 0
    current_run = 0
    for flag in underwater:
        if flag:
            current_run += 1
            if current_run > max_run:
                max_run = current_run
        else:
            current_run = 0
    return int(max_run)


def _strategy_max_drawdown(account_history: pd.DataFrame) -> float:
    """现金流调整后策略净值回撤：剔除新增本金的影响。"""
    if account_history.empty:
        return np.nan
    data = account_history.copy()
    data["net_value"] = (
        data["account_asset"] - data["cumulative_contribution"].shift(1).fillna(0)
    )
    # 起点净值为 0，无法直接计算回撤，使用累计净流入作为基线
    data["strategy_nav"] = data["net_value"].cummax().where(
        data["net_value"] > 0, other=data["net_value"]
    )
    data["strategy_drawdown"] = (
        data["strategy_nav"] / data["strategy_nav"].cummax() - 1.0
    )
    return float(data["strategy_drawdown"].min())


@dataclass
class BacktestOutput:
    summary: dict[str, Any]
    transactions: pd.DataFrame
    account_history: pd.DataFrame
    total_return_history: pd.DataFrame


def simulate_bank_dca(
    *,
    symbol: str,
    name: str,
    prices: pd.DataFrame,
    dividends: pd.DataFrame,
    listing_date: Any,
    as_of_date: str,
    horizon_years: int,
    monthly_amount: float = 5000.0,
    buy_day: int = 1,
    lot_size: int = 100,
    dividend_reinvest: bool = True,
) -> BacktestOutput:
    """单只银行月度定投回测。

    纯计算函数：输入不复权行情 + 已实施分红 DataFrame，输出交易流水、
    账户历史、标的总收益净值与汇总指标。不调用任何外部接口。

    回测口径（与本 Lab .md 一致）：
    - 每月投入固定金额，不足 1 手的零钱留现金；
    - 除权日按收盘价立即用分红再投资原标的；
    - 不计算手续费与分红税；
    - 上市晚于窗口起点或数据不完整时，比较指标返回 np.nan。
    """
    symbol = normalize_symbol(symbol)
    as_of = pd.Timestamp(as_of_date)
    requested_start = as_of - pd.DateOffset(years=horizon_years)
    listing = pd.Timestamp(listing_date)

    price = prices.copy()
    price["date"] = pd.to_datetime(price["date"])
    price = price[
        price["date"].le(as_of) & price["date"].ge(min(requested_start, listing))
    ].drop_duplicates("date", keep="last").sort_values("date")
    if price.empty:
        raise ValueError(f"{symbol} 在 {horizon_years} 年窗口内无行情")

    actual_start_target = max(requested_start, listing)
    available = price[price["date"].ge(actual_start_target)]
    if available.empty:
        raise ValueError(f"{symbol} 上市后至截止日无行情")
    start = pd.Timestamp(available["date"].iloc[0])
    end = pd.Timestamp(price["date"].max())
    price = price[price["date"].between(start, end)].reset_index(drop=True)
    full_horizon = bool(
        listing <= requested_start
        and start <= requested_start + pd.Timedelta(days=15)
    )

    schedule = set(
        contribution_dates(
            price["date"],
            start_date=start,
            end_date=end,
            buy_day=buy_day,
        )
    )
    dividend_events = dividends.copy()
    if dividend_events.empty:
        dividend_events = pd.DataFrame(
            columns=["record_date", "ex_date", "cash_dividend_per_share"]
        )
    dividend_events["record_date"] = pd.to_datetime(
        dividend_events["record_date"], errors="coerce"
    )
    dividend_events["ex_date"] = pd.to_datetime(
        dividend_events["ex_date"], errors="coerce"
    )
    dividend_events = dividend_events[
        dividend_events["ex_date"].between(start, end)
    ].copy()
    grouped_dividends = {
        date: group for date, group in dividend_events.groupby("ex_date")
    }

    cash = 0.0
    shares = 0
    cumulative_contribution = 0.0
    total_dividend = 0.0
    total_purchase_cost = 0.0
    snapshots: dict[pd.Timestamp, int] = {}
    transactions: list[dict[str, Any]] = []
    account_rows: list[dict[str, Any]] = []
    external_cashflows: list[float] = []
    external_dates: list[pd.Timestamp] = []

    def execute_buy(date: pd.Timestamp, close: float, trade_type: str) -> None:
        nonlocal cash, shares, total_purchase_cost
        buy_shares = int(cash // (close * lot_size)) * lot_size
        buy_amount = buy_shares * close
        cash -= buy_amount
        shares += buy_shares
        total_purchase_cost += buy_amount
        transactions.append(
            {
                "symbol": symbol,
                "name": name,
                "horizon": f"{horizon_years}Y",
                "date": date,
                "trade_type": trade_type,
                "buy_price": close,
                "buy_shares": buy_shares,
                "buy_amount": buy_amount,
                "fee": 0.0,
                "remaining_cash": cash,
                "cumulative_shares": shares,
                "cumulative_contribution": cumulative_contribution,
                "repo_interest_accrued": 0.0,
                "dividend_received": 0.0,
                "dividend_tax": 0.0,
            }
        )

    for row in price.itertuples(index=False):
        date = pd.Timestamp(row.date)
        close = float(row.close)
        dividend_received_today = 0.0

        if date in grouped_dividends:
            for event in grouped_dividends[date].itertuples(index=False):
                record_date = (
                    pd.Timestamp(event.record_date)
                    if pd.notna(event.record_date)
                    else date - pd.Timedelta(days=1)
                )
                eligible_shares = _shares_on_or_before(snapshots, record_date)
                dividend_cash = (
                    eligible_shares * float(event.cash_dividend_per_share)
                )
                cash += dividend_cash
                total_dividend += dividend_cash
                dividend_received_today += dividend_cash
            if dividend_reinvest and dividend_received_today > 0:
                execute_buy(date, close, "dividend_reinvest")
                transactions[-1]["dividend_received"] = dividend_received_today

        if date in schedule:
            cash += monthly_amount
            cumulative_contribution += monthly_amount
            external_cashflows.append(-monthly_amount)
            external_dates.append(date)
            execute_buy(date, close, "monthly_contribution")

        snapshots[date] = shares
        market_value = shares * close
        asset = market_value + cash
        account_profit_rate = (
            asset / cumulative_contribution - 1.0
            if cumulative_contribution > 0 else np.nan
        )
        account_rows.append(
            {
                "symbol": symbol,
                "name": name,
                "horizon": f"{horizon_years}Y",
                "date": date,
                "close": close,
                "shares": shares,
                "cash": cash,
                "market_value": market_value,
                "account_asset": asset,
                "cumulative_contribution": cumulative_contribution,
                "account_profit_rate": account_profit_rate,
                "dividend_received": dividend_received_today,
            }
        )

    account = pd.DataFrame(account_rows)
    transaction_frame = pd.DataFrame(transactions)
    ending_market_value = float(account["market_value"].iloc[-1])
    ending_asset = float(account["account_asset"].iloc[-1])
    ending_cash = float(account["cash"].iloc[-1])
    ending_shares = int(account["shares"].iloc[-1])
    external_cashflows.append(ending_asset)
    external_dates.append(end)
    annualized_return = xirr(external_cashflows, external_dates)

    total_history = build_total_return_history(price, dividend_events)
    volatility = float(
        total_history["daily_total_return"].iloc[1:].std(ddof=1) * math.sqrt(252)
    )
    average_buy_price = (
        total_purchase_cost / ending_shares if ending_shares else np.nan
    )
    current_profit_rate = (
        float(price["close"].iloc[-1]) / average_buy_price - 1.0
        if average_buy_price and not np.isnan(average_buy_price) else np.nan
    )
    total_return = (
        ending_asset / cumulative_contribution - 1.0
        if cumulative_contribution else np.nan
    )
    max_loss = float(account["account_profit_rate"].min())
    max_drawdown = float(total_history["drawdown"].min())
    max_loss_duration = _max_loss_duration_days(account)
    strategy_max_dd = _strategy_max_drawdown(account)

    summary = {
        "symbol": symbol,
        "name": name,
        "horizon": f"{horizon_years}Y",
        "requested_start_date": requested_start,
        "start_date": start,
        "end_date": end,
        "listing_date": listing,
        "full_horizon": full_horizon,
        "contribution_months": len(schedule),
        "total_contribution": cumulative_contribution,
        "ending_shares": ending_shares,
        "ending_cash": ending_cash,
        "ending_market_value": ending_market_value,
        "ending_asset": ending_asset,
        "total_dividend": total_dividend,
        "total_return": total_return if full_horizon else np.nan,
        "xirr": annualized_return if full_horizon else np.nan,
        "average_buy_price": average_buy_price if full_horizon else np.nan,
        "current_profit_rate": current_profit_rate if full_horizon else np.nan,
        "max_drawdown": max_drawdown if full_horizon else np.nan,
        "max_loss_vs_contribution": max_loss if full_horizon else np.nan,
        "strategy_max_drawdown": strategy_max_dd if full_horizon else np.nan,
        "max_loss_duration_days": max_loss_duration if full_horizon else np.nan,
        "volatility": volatility if full_horizon else np.nan,
        "dividend_reinvest": dividend_reinvest,
    }
    return BacktestOutput(
        summary=summary,
        transactions=transaction_frame,
        account_history=account,
        total_return_history=total_history.assign(
            symbol=symbol, name=name, horizon=f"{horizon_years}Y"
        ),
    )

## 3. 读取 Task 1 标准表

In [ ]:
security_info = pd.read_csv(DATA_DIR / "security_info.csv", parse_dates=["list_date", "manual_review_as_of"])
symbol_info = security_info[security_info["symbol"] == SYMBOL].iloc[0]

prices = pd.read_parquet(PRICE_DIR / f"{SYMBOL}_daily_raw.parquet")
prices["date"] = pd.to_datetime(prices["date"])

dividend_path = DIVIDEND_DIR / f"{SYMBOL}_dividend.parquet"
dividends = pd.read_parquet(dividend_path) if dividend_path.is_file() else pd.DataFrame(
    columns=["ex_date", "record_date", "cash_dividend_per_share"]
)
if not dividends.empty:
    dividends["ex_date"] = pd.to_datetime(dividends["ex_date"])
    dividends["record_date"] = pd.to_datetime(dividends["record_date"])

print(f"行情: {len(prices)} 行, 分红: {len(dividends)} 条")
print(f"上市日期: {symbol_info['list_date']}")

## 4. 执行回测

In [ ]:
outputs = {}
for horizon in HORIZONS:
    output = simulate_bank_dca(
        symbol=SYMBOL,
        name=NAME,
        prices=prices,
        dividends=dividends,
        listing_date=symbol_info["list_date"],
        as_of_date=AS_OF_DATE,
        horizon_years=horizon,
        monthly_amount=MONTHLY_AMOUNT,
        buy_day=BUY_DAY,
        lot_size=LOT_SIZE,
        dividend_reinvest=DIVIDEND_REINVEST,
    )
    outputs[horizon] = output
    txn_path = TRANSACTION_DIR / f"{SYMBOL}_{horizon}Y_no_fee.csv"
    output.transactions.to_csv(txn_path, index=False, encoding="utf-8-sig")
    print(f"{horizon}Y: 投入 {output.summary['total_contribution']:.0f} 元, "
          f"期末资产 {output.summary['ending_asset']:.0f} 元, "
          f"XIRR={output.summary['xirr']:.4f}, "
          f"完整窗口={output.summary['full_horizon']}")

## 5. 指标汇总

In [ ]:
summary_rows = []
for horizon, output in outputs.items():
    summary_rows.append(output.summary)
summary_df = pd.DataFrame(summary_rows)
display(summary_df[[
    "horizon", "full_horizon", "xirr", "total_return",
    "strategy_max_drawdown", "max_loss_vs_contribution",
    "max_loss_duration_days", "total_contribution", "ending_asset",
    "total_dividend", "ending_shares", "ending_cash", "volatility",
]])

## 6. 图表

5 张图：总回报净值路径、账户资产 vs 累计投入、月度投入流水、回撤分布、分红贡献。

In [ ]:
# 图1：总回报净值路径（10Y，起点=100）
output_10y = outputs[10]
tr = output_10y.total_return_history.copy()

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(tr["date"], tr["total_return_nav"], label="标的总回报净值", color="steelblue", linewidth=1.5)
ax.axhline(100, color="gray", linestyle="--", alpha=0.5)
ax.set_title(f"{NAME} 10Y 标的总回报净值（起点=100）")
ax.set_xlabel("日期")
ax.set_ylabel("净值")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(CHART_DIR / "01_total_return_nav.png", dpi=120)
plt.show()

In [ ]:
# 图2：账户资产 vs 累计投入（10Y）
account = output_10y.account_history

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(account["date"], account["account_asset"], label="账户资产", color="steelblue")
ax.plot(account["date"], account["cumulative_contribution"], label="累计投入", color="orange", linestyle="--")
ax.fill_between(account["date"], account["account_asset"], account["cumulative_contribution"],
                where=account["account_asset"] >= account["cumulative_contribution"],
                alpha=0.2, color="green", label="浮盈")
ax.fill_between(account["date"], account["account_asset"], account["cumulative_contribution"],
                where=account["account_asset"] < account["cumulative_contribution"],
                alpha=0.2, color="red", label="浮亏")
ax.set_title(f"{NAME} 10Y 账户资产 vs 累计投入")
ax.set_xlabel("日期")
ax.set_ylabel("金额（元）")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(CHART_DIR / "02_account_vs_contribution.png", dpi=120)
plt.show()

In [ ]:
# 图3：月度投入流水（10Y，前 36 个月）
txn = output_10y.transactions
buys = txn[txn["type"] == "buy"].head(36)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(buys["date"], buys["amount"], color="steelblue", label="月度买入")
ax.set_title(f"{NAME} 10Y 月度投入流水（前 36 个月）")
ax.set_xlabel("日期")
ax.set_ylabel("买入金额（元）")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(CHART_DIR / "03_monthly_buy_flow.png", dpi=120)
plt.show()

In [ ]:
# 图4：回撤分布（10Y，标的总回报净值回撤）
tr_nav = tr.set_index("date")["total_return_nav"]
peak = tr_nav.cummax()
drawdown = (tr_nav - peak) / peak * 100

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(drawdown.index, drawdown.values, 0, color="red", alpha=0.3)
ax.plot(drawdown.index, drawdown.values, color="red", linewidth=0.8)
ax.set_title(f"{NAME} 10Y 标的总回报净值回撤")
ax.set_xlabel("日期")
ax.set_ylabel("回撤（%）")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(CHART_DIR / "04_drawdown.png", dpi=120)
plt.show()

In [ ]:
# 图5：分红贡献（10Y，累计分红 vs 累计买入）
account_10y = output_10y.account_history

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(account_10y["date"], account_10y["cumulative_dividend"], label="累计分红", color="green")
ax.plot(account_10y["date"], account_10y["cumulative_buy_cost"], label="累计买入成本", color="steelblue")
ax.set_title(f"{NAME} 10Y 分红贡献")
ax.set_xlabel("日期")
ax.set_ylabel("金额（元）")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(CHART_DIR / "05_dividend_contribution.png", dpi=120)
plt.show()

## 7. 验收

In [ ]:
assert len(summary_df) == len(HORIZONS), f"应生成 {len(HORIZONS)} 档结果"
for horizon in HORIZONS:
    assert horizon in outputs, f"{horizon}Y 结果缺失"
chart_files = list(CHART_DIR.glob("*.png"))
assert len(chart_files) >= 5, f"应生成 5 张图，实际 {len(chart_files)} 张"
print(f"Task 2 验收通过")
print(f"- {len(HORIZONS)} 档周期回测完成")
print(f"- {len(chart_files)} 张图表生成")
print(f"- 交易流水写入: {TRANSACTION_DIR.relative_to(LABS_DIR)}")